# Práctica 1: Segmentación de Clientes mediante RFM y K-Means

## 1. Contexto
En esta práctica trabajaremos con el dataset público **Online Retail** del *UCI Machine Learning Repository* (DOI: `10.24432/C5BW33`, Licencia CC BY 4.0). Contiene **541.909 líneas de transacción** registradas entre diciembre de 2010 y diciembre de 2011 por una empresa británica de comercio electrónico.

**Variables principales:**
- `InvoiceNo`: identificador de factura (los valores que empiezan por `C` indican cancelaciones).
- `Quantity`: unidades adquiridas.
- `InvoiceDate`: fecha y hora de la transacción.
- `UnitPrice`: precio unitario.
- `CustomerID`: identificador del cliente.
- `Country`: país del cliente.

**Objetivo:** Transformar las transacciones en un dataset de clientes mediante variables **RFM** (*Recency, Frequency, Monetary*) y aplicar **K-Means** para identificar segmentos comercialmente interpretables.

### 1.1. Importación de librerías y carga del dataset

In [5]:
# Importación de librerías principales
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA

# Configuración visual
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

In [6]:
# Carga del dataset Online Retail (desde memoria, archivo local o UCI Repository)
import importlib, subprocess, sys
try:
    import openpyxl
except ImportError:
    print('Instalando openpyxl en el entorno actual...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl'])
    importlib.invalidate_caches()
    import openpyxl

UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
LOCAL_XLSX = 'Online Retail.xlsx'
LOCAL_CSV = 'Online_Retail.csv'

if 'df' in globals() and isinstance(df, pd.DataFrame) and not df.empty:
    print('El dataset ya está cargado en memoria.')
elif os.path.exists(LOCAL_CSV):
    print(f'Cargando desde archivo CSV local: {LOCAL_CSV}...')
    df = pd.read_csv(LOCAL_CSV, encoding='ISO-8859-1', parse_dates=['InvoiceDate'])
elif os.path.exists(LOCAL_XLSX):
    print(f'Cargando desde archivo Excel local: {LOCAL_XLSX}...')
    df = pd.read_excel(LOCAL_XLSX)
else:
    print('Descargando dataset desde UCI Machine Learning Repository...')
    df = pd.read_excel(UCI_URL)
    try:
        df.to_csv(LOCAL_CSV, index=False, encoding='ISO-8859-1')
        print(f'Copia guardada localmente en {LOCAL_CSV}')
    except Exception:
        pass

print(f'Dimensiones del dataset: {df.shape[0]:,} filas y {df.shape[1]} columnas')
df.head()

El dataset ya está cargado en memoria.
Dimensiones del dataset: 541,909 filas y 8 columnas


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [7]:
# Información general de tipos de datos y valores nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


### Descripción detallada de las columnas del dataset

A partir de la salida de `df.info()` y de la documentación oficial del dataset **Online Retail** (541.909 registros y 8 columnas), se detalla el significado, tipo de dato y particularidades de cada variable:

| Columna | Tipo de dato | Valores no nulos | Descripción y relevancia para el análisis RFM |
| :--- | :--- | :--- | :--- |
| **`InvoiceNo`** | `object` (alfanumérico) | 541.909 (sin nulos) | **Número de factura.** Identificador único de 6 dígitos asignado a cada transacción/pedido. Si el código comienza por la letra **`'C'`** (ej. `C536379`), indica una **cancelación o devolución**. Es clave para filtrar pedidos anulados y calcular la **Frecuencia (*Frequency*)** contando facturas únicas por cliente. |
| **`StockCode`** | `object` (alfanumérico) | 541.909 (sin nulos) | **Código de producto/artículo.** Identificador único de 5 dígitos asignado a cada producto distinto del catálogo (incluye algunos códigos especiales como `POST` de gastos de envío o `M` de ajustes manuales). |
| **`Description`** | `object` (texto) | 540.455 (~1.454 nulos) | **Descripción comercial del producto.** Nombre del artículo adquirido. Presenta algunos valores nulos en ajustes de inventario, pero no afecta directamente al cálculo de las métricas RFM. |
| **`Quantity`** | `int64` (entero) | 541.909 (sin nulos) | **Cantidad de unidades** de cada producto por línea de transacción. Toma valores negativos en devoluciones/cancelaciones o ajustes de stock, por lo que será necesario conservar únicamente `Quantity > 0` para analizar compras reales. |
| **`InvoiceDate`** | `datetime64[ns]` | 541.909 (sin nulos) | **Fecha y hora de la factura.** Día y hora exacta en que se generó la transacción (entre el 01/12/2010 y el 09/12/2011). Es imprescindible para calcular la **Recencia (*Recency*)**: días transcurridos desde la última compra de cada cliente hasta la fecha de referencia. |
| **`UnitPrice`** | `float64` (decimal) | 541.909 (sin nulos) | **Precio unitario** del producto en libras esterlinas (£). Multiplicado por `Quantity` (`Quantity * UnitPrice`) permite obtener el importe total de cada línea para calcular el **Valor Monetario (*Monetary*)** de cada cliente. Deben filtrarse valores `<= 0`. |
| **`CustomerID`** | `float64` (numérico) | 406.829 (~135.080 nulos) | **Identificador único del cliente** (número de 5 dígitos). Aproximadamente un **24,9 %** de las filas no tienen `CustomerID` (compras de usuarios no registrados o ventas sin ficha). Al ser una segmentación a nivel de cliente, será obligatorio **eliminar las filas sin `CustomerID`** y convertirlo a entero/texto. |
| **`Country`** | `object` (texto) | 541.909 (sin nulos) | **País de residencia** del cliente donde se realiza el pedido (mayoritariamente *United Kingdom*, seguido de otros países europeos y del resto del mundo). |

---
## 2. Exploración inicial
Analizamos el dataset transaccional antes de agregar la información por cliente, desglosando cada punto en una celda independiente.

### 2.1. Número de filas, columnas y periodo temporal cubierto

In [ ]:
# 2.1. Número de filas, columnas y periodo temporal cubierto
n_filas, n_columnas = df.shape
fecha_min = pd.to_datetime(df['InvoiceDate']).min()
fecha_max = pd.to_datetime(df['InvoiceDate']).max()
duracion = fecha_max - fecha_min

print(f'Número total de filas:    {n_filas:,}')
print(f'Número total de columnas: {n_columnas} -> {list(df.columns)}')
print(f'Fecha primera transacción: {fecha_min}')
print(f'Fecha última transacción:  {fecha_max}')
print(f'Periodo temporal cubierto: {duracion.days} días (~{duracion.days / 30.44:.1f} meses)')

### 2.2. Tipos de datos y conversión de `InvoiceDate`

In [ ]:
# 2.2. Tipos de datos y conversión explícita de InvoiceDate a datetime
print('--- Tipos de datos originales ---')
print(df.dtypes)

# Aseguramos que InvoiceDate sea datetime64 y que InvoiceNo / StockCode sean texto para evaluar prefijos como "C"
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceNo'] = df['InvoiceNo'].astype(str).str.strip()
df['StockCode'] = df['StockCode'].astype(str).str.strip()

print('\n--- Tipos de datos tras la conversión ---')
print(df.dtypes)

### 2.3. Valores nulos, especialmente en `CustomerID`

In [ ]:
# 2.3. Estudio de valores nulos por columna (recuento y porcentaje)
nulos_abs = df.isnull().sum()
nulos_pct = (df.isnull().mean() * 100)

resumen_nulos = pd.DataFrame({
    'Valores Nulos': nulos_abs,
    'Porcentaje (%)': nulos_pct,
    'Valores No Nulos': df.notnull().sum()
}).sort_values(by='Valores Nulos', ascending=False)

print(f"Filas sin CustomerID: {nulos_abs['CustomerID']:,} ({nulos_pct['CustomerID']:.2f}% del total)")
print(f"Filas con CustomerID válido: {df['CustomerID'].notnull().sum():,} ({100 - nulos_pct['CustomerID']:.2f}% del total)")
resumen_nulos

### 2.4. Facturas canceladas y cantidades o precios no positivos

In [ ]:
# 2.4. Análisis de facturas canceladas (empiezan por 'C') y cantidades o precios <= 0
mask_canceladas = df['InvoiceNo'].str.startswith('C', na=False)
mask_qty_no_pos = df['Quantity'] <= 0
mask_price_no_pos = df['UnitPrice'] <= 0
mask_price_neg = df['UnitPrice'] < 0
mask_price_zero = df['UnitPrice'] == 0

print(f'1) Facturas canceladas (InvoiceNo empieza por "C"):')
print(f'   - Líneas de transacción canceladas: {mask_canceladas.sum():,} ({mask_canceladas.mean()*100:.2f}%)')
print(f'   - Número de facturas únicas canceladas: {df.loc[mask_canceladas, "InvoiceNo"].nunique():,}\n')

print(f'2) Cantidades no positivas (Quantity <= 0):')
print(f'   - Total de líneas con Quantity <= 0: {mask_qty_no_pos.sum():,} ({mask_qty_no_pos.mean()*100:.2f}%)')
print(f'   - De ellas, pertenecen a facturas canceladas ("C"): {(mask_qty_no_pos & mask_canceladas).sum():,}')
print(f'   - Sin prefijo "C" (ajustes/mermas de inventario sin CustomerID): {(mask_qty_no_pos & ~mask_canceladas).sum():,}\n')

print(f'3) Precios unitarios no positivos (UnitPrice <= 0):')
print(f'   - Total de líneas con UnitPrice <= 0: {mask_price_no_pos.sum():,} ({mask_price_no_pos.mean()*100:.2f}%)')
print(f'   - Líneas con UnitPrice == 0 (regalos/ajustes): {mask_price_zero.sum():,}')
print(f'   - Líneas con UnitPrice < 0 (deuda incobrable / Adjust bad debt): {mask_price_neg.sum():,}')

### 2.5. Número de clientes, facturas, productos y países distintos

In [ ]:
# 2.5. Cardinalidad de entidades únicas en el dataset bruto
n_clientes = df['CustomerID'].nunique(dropna=True)
n_facturas_total = df['InvoiceNo'].nunique()
n_facturas_validas = df.loc[~df['InvoiceNo'].str.startswith('C'), 'InvoiceNo'].nunique()
n_productos_codigo = df['StockCode'].nunique()
n_productos_desc = df['Description'].nunique(dropna=True)
n_paises = df['Country'].nunique()

print(f'Clientes distintos (CustomerID válidos): {n_clientes:,}')
print(f'Facturas distintas totales (InvoiceNo):  {n_facturas_total:,} (de las cuales {n_facturas_validas:,} no son cancelaciones)')
print(f'Productos distintos por StockCode:       {n_productos_codigo:,}')
print(f'Descripciones de producto distintas:     {n_productos_desc:,}')
print(f'Países distintos (Country):              {n_paises:,}\n')

print('Top 10 países por número de líneas de transacción:')
top_paises = df['Country'].value_counts().head(10).to_frame(name='Transacciones')
top_paises['Porcentaje (%)'] = (top_paises['Transacciones'] / len(df)) * 100
top_paises

### 2.6. Distribución del importe de las transacciones y posibles valores extremos

In [ ]:
# 2.6. Cálculo del importe por línea (Quantity * UnitPrice), estadísticos y detección de valores extremos
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print('--- Estadísticos descriptivos de Quantity, UnitPrice y TotalPrice (importe por línea) ---')
percent_list = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
display(df[['Quantity', 'UnitPrice', 'TotalPrice']].describe(percentiles=percent_list).T)

print('\n--- Top 5 transacciones con MAYOR importe positivo (outliers superiores) ---')
display(df.nlargest(5, 'TotalPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TotalPrice', 'CustomerID', 'Country']])

print('\n--- Top 5 transacciones con MENOR importe (cancelaciones / ajustes extremos) ---')
display(df.nsmallest(5, 'TotalPrice')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TotalPrice', 'CustomerID', 'Country']])

# Visualización de la distribución y valores extremos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Boxplot de TotalPrice evidenciando los valores extremos
sns.boxplot(x=df['TotalPrice'], ax=axes[0], color='salmon')
axes[0].set_title('Boxplot del Importe por Línea (con valores extremos)')
axes[0].set_xlabel('Importe de la línea (£)')

# Gráfico 2: Histograma del importe acotado entre percentiles 1% y 99% (compras normales)
p01, p99 = df['TotalPrice'].quantile([0.01, 0.99])
sns.histplot(df.loc[df['TotalPrice'].between(p01, p99), 'TotalPrice'], bins=60, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title(f'Distribución del Importe por Línea (Percentiles 1% a 99%: [{p01:.1f}£, {p99:.1f}£])')
axes[1].set_xlabel('Importe de la línea (£)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

### 2.7. Pregunta de análisis: ¿Por qué cada fila del archivo representa una línea de factura y no un cliente completo?

**Respuesta:**

Cada fila del dataset original tiene una **granularidad de línea de pedido (*order line-item*)** porque registra la compra de **un producto concreto (`StockCode`) dentro de una factura determinada (`InvoiceNo`)** realizada por un cliente (`CustomerID`) en una fecha específica (`InvoiceDate`).

En un sistema transaccional de comercio electrónico (OLTP) existe una relación jerárquica **1 a muchos ($1:N$)** en dos niveles:
1. **Un cliente (`CustomerID`) puede realizar múltiples compras/facturas (`InvoiceNo`)** a lo largo del año (por ejemplo, comprar en diciembre, marzo y noviembre).
2. **Una misma factura (`InvoiceNo`) suele incluir varios artículos distintos (`StockCode`)** en el carrito de la compra, generando una fila por cada producto diferente adquirido en ese pedido.

Por tanto, un mismo `CustomerID` aparece repetido en decenas o cientos de filas. Si aplicáramos K-Means directamente sobre este archivo sin agregarlo antes, estaríamos agrupando **líneas de productos sueltos** en lugar de **clientes**, distorsionando completamente la segmentación. Por ello es imprescindible agregar primero las transacciones por `CustomerID` mediante las métricas **RFM** (*Recency, Frequency, Monetary*), de modo que cada fila de la nueva tabla represente exactamente a **un único cliente**.

In [ ]:
# Demostración práctica: ejemplo del cliente 17850 y la factura 536365
ejemplo_factura = df[df['InvoiceNo'] == '536365']
ejemplo_cliente = df[df['CustomerID'] == 17850]

print(f'La factura 536365 (de un único pedido) ocupa {len(ejemplo_factura)} filas (una por producto del carrito).')
print(f'El cliente 17850 aparece en {len(ejemplo_cliente)} filas repartidas en {ejemplo_cliente["InvoiceNo"].nunique()} facturas distintas.')

---
## 3. Preparación y construcción del dataset RFM

Para transformar el registro de líneas transaccionales en un dataset con **una única fila por cliente**, aplicamos un pipeline secuencial y trazable de limpieza, agregación, análisis de asimetrías y estandarización:

1. **Eliminar registros sin `CustomerID`**: sin identificador único es imposible atribuir la compra a un cliente concreto para calcular su historial RFM (representan un 24,93 % de las líneas, correspondientes a compras de invitados o ajustes internos).
2. **Excluir facturas canceladas (`InvoiceNo` que empieza por `'C'`) y registros con `Quantity <= 0` o `UnitPrice <= 0`**: las cancelaciones, devoluciones, mermas de almacén, regalos promocionales (`UnitPrice == 0`) y apuntes contables (`Adjust bad debt`) no representan compras efectivas de productos y distorsionarían tanto la frecuencia de compra como el gasto total.
3. **Calcular `total_amount = Quantity * UnitPrice`**: importe monetario neto de cada línea de factura válida.
4. **Definir una fecha de referencia (`reference_date`) posterior a la última transacción**: tomamos `max(InvoiceDate) + 1 día` (`2011-12-10 12:50:00`) para asegurar que los clientes que compraron el último día registrado tengan `recency_days >= 1`.
5. **Agregar por `CustomerID`** para construir las 5 variables de comportamiento del cliente (`recency_days`, `purchase_frequency`, `monetary_value`, `average_order_value`, `total_units`) y exportar a `online_retail_rfm.csv`.

### 3.1. Limpieza de datos transaccionales y cálculo de `total_amount` (Pasos 1, 2 y 3)

In [ ]:
# Copia de trabajo para mantener intacto el dataframe original
df_clean = df.copy()
n_inicial = len(df_clean)

# Paso 1: Eliminar registros sin CustomerID
df_clean = df_clean.dropna(subset=['CustomerID']).copy()
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
n_tras_customer = len(df_clean)

# Paso 2a: Excluir facturas canceladas (InvoiceNo comienza por 'C')
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str).str.strip()
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C', na=False)].copy()
n_tras_cancel = len(df_clean)

# Paso 2b: Excluir registros con Quantity <= 0 o UnitPrice <= 0
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)].copy()
n_final = len(df_clean)

# Paso 3: Calcular total_amount = Quantity * UnitPrice
df_clean['total_amount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print('=== RESUMEN DE TRAZABILIDAD DE LA LIMPIEZA ===')
print(f'1. Registros iniciales:                         {n_inicial:>9,} (100.00%)')
print(f'2. Tras eliminar nulos en CustomerID:           {n_tras_customer:>9,} ({n_tras_customer/n_inicial*100:>6.2f}%) -> eliminados: {n_inicial - n_tras_customer:,}')
print(f'3. Tras excluir cancelaciones (prefijo "C"):    {n_tras_cancel:>9,} ({n_tras_cancel/n_inicial*100:>6.2f}%) -> eliminados: {n_tras_customer - n_tras_cancel:,}')
print(f'4. Tras exigir Quantity > 0 y UnitPrice > 0:    {n_final:>9,} ({n_final/n_inicial*100:>6.2f}%) -> eliminados: {n_tras_cancel - n_final:,}')
print(f'\nTotal de líneas válidas para construir RFM:     {n_final:,} filas de {df_clean["CustomerID"].nunique():,} clientes únicos.')
df_clean.head()

### 3.2. Fecha de referencia, agregación por cliente y guardado de `online_retail_rfm.csv` (Pasos 4 y 5)

In [ ]:
# Paso 4: Definir fecha de referencia (1 día después de la última transacción registrada)
max_invoice_date = pd.to_datetime(df_clean['InvoiceDate']).max()
reference_date = max_invoice_date + pd.Timedelta(days=1)

print(f'Última transacción válida: {max_invoice_date}')
print(f'Fecha de referencia RFM:   {reference_date}')

# Paso 5: Agregar datos por cliente para obtener las variables requeridas
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
rfm = df_clean.groupby('CustomerID', as_index=False).agg(
    recency_days=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    purchase_frequency=('InvoiceNo', 'nunique'),
    monetary_value=('total_amount', 'sum'),
    total_units=('Quantity', 'sum')
)

# Renombramos CustomerID a customer_id y calculamos average_order_value
rfm = rfm.rename(columns={'CustomerID': 'customer_id'})
rfm['average_order_value'] = rfm['monetary_value'] / rfm['purchase_frequency']

# Ordenamos las columnas exactamente según las especificaciones del enunciado
rfm_cols = [
    'customer_id',
    'recency_days',
    'purchase_frequency',
    'monetary_value',
    'average_order_value',
    'total_units'
]
rfm = rfm[rfm_cols]

# Guardamos el dataset como online_retail_rfm.csv para conservar la trazabilidad
csv_rfm_path = 'online_retail_rfm.csv'
try:
    rfm.to_csv(csv_rfm_path, index=False)
    print(f'\nArchivo guardado correctamente en: {csv_rfm_path}')
except Exception:
    import tempfile
    alt_path = os.path.join(tempfile.gettempdir(), 'online_retail_rfm.csv')
    rfm.to_csv(alt_path, index=False)
    print(f'\nArchivo guardado en ruta temporal: {alt_path}')

print(f'Dimensiones del dataset RFM de clientes: {rfm.shape[0]:,} clientes x {rfm.shape[1]} columnas')
display(rfm.head(10))

### 3.3. Estudio de asimetrías y valores extremos antes de transformar o escalar (Paso 6)

Analizamos la distribución estadística, los percentiles y el **coeficiente de asimetría (*skewness*)** de las 5 variables numéricas construidas a nivel de cliente.

In [ ]:
# Variables numéricas de comportamiento del cliente
feature_cols = [
    'recency_days',
    'purchase_frequency',
    'monetary_value',
    'average_order_value',
    'total_units'
]

# Tabla de estadísticos descriptivos + coeficiente de asimetría (skewness)
stats_rfm = rfm[feature_cols].describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
stats_rfm['skewness'] = rfm[feature_cols].skew()
print('--- Estadísticos descriptivos y coeficiente de asimetría (Skewness) de las variables RFM ---')
display(stats_rfm[['count', 'mean', 'std', 'min', '25%', '50%', '75%', '95%', '99%', 'max', 'skewness']])

# Visualización de histogramas y boxplots de las variables originales
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for i, col in enumerate(feature_cols):
    sns.histplot(rfm[col], bins=40, kde=True, ax=axes[0, i], color='steelblue')
    axes[0, i].set_title(f'Distribución: {col}\n(Skew = {rfm[col].skew():.2f})', fontsize=10)
    axes[0, i].set_xlabel('')
    
    sns.boxplot(x=rfm[col], ax=axes[1, i], color='salmon')
    axes[1, i].set_title(f'Boxplot: {col}', fontsize=10)
    axes[1, i].set_xlabel('')

plt.suptitle('Distribución original y valores extremos de las variables RFM (antes de transformar)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.4. Transformación logarítmica y estandarización de las variables (Paso 7)

**Justificación del preprocesamiento previo a K-Means:**
- Como muestran los estadísticos anteriores, variables como `monetary_value` ($\text{skewness} \approx 19$), `total_units` ($\text{skewness} \approx 19$), `average_order_value` ($\text{skewness} \approx 34$) y `purchase_frequency` ($\text{skewness} \approx 12$) presentan una **asimetría positiva extrema** causada por clientes mayoristas que compran decenas de miles de libras.
- Antes de estandarizar, aplicamos la transformación monótona **`np.log1p(x)`** ($\ln(1+x)$) para comprimir la cola derecha y estabilizar la varianza sin eliminar clientes reales de alto valor.
- A continuación, aplicamos **`StandardScaler`** ($z = \frac{x - \mu}{\sigma}$) para que todas las variables tengan **media $0$ y desviación típica $1$**, garantizando que ninguna variable domine el cálculo de distancias euclídeas en K-Means por su escala numérica.

In [ ]:
# Paso 7a: Transformación logarítmica (log1p) para corregir la fuerte asimetría positiva
rfm_log = np.log1p(rfm[feature_cols])

# Paso 7b: Estandarización (media = 0, desviación estándar = 1) con StandardScaler
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(rfm_log)
X_scaled = pd.DataFrame(X_scaled_array, columns=feature_cols, index=rfm.index)

print('--- Verificación de media (≈0) y desviación estándar (≈1) tras log1p + StandardScaler ---')
resumen_escalado = X_scaled.describe().T[['mean', 'std', 'min', '50%', 'max']]
resumen_escalado['skewness_post_log'] = X_scaled.skew()
display(resumen_escalado)

# Visualización de las distribuciones tras transformación logarítmica y escalado
fig, axes = plt.subplots(1, 5, figsize=(20, 3.8))
for i, col in enumerate(feature_cols):
    sns.histplot(X_scaled[col], bins=35, kde=True, ax=axes[i], color='seagreen')
    axes[i].set_title(f'{col} (Escalado)\nSkew = {X_scaled[col].skew():.2f}', fontsize=10)
    axes[i].set_xlabel('Z-score')

plt.suptitle('Distribuciones tras transformación logarítmica (log1p) y StandardScaler', fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

### 3.5. Pregunta de análisis: ¿Qué ocurriría si aplicáramos K-Means directamente sobre variables monetarias y de frecuencia sin transformar ni escalar?

**Respuesta:**

El algoritmo **K-Means** agrupa observaciones minimizando la suma de distancias euclídeas al cuadrado respecto a los centroides de cada cluster:

$$d(\mathbf{x}_i, \boldsymbol{\mu}_c) = \sqrt{\sum_{j=1}^{p} (x_{ij} - \mu_{cj})^2}$$

Si aplicáramos K-Means directamente sobre las variables en bruto sin transformar ni escalar, ocurrirían **dos problemas graves**:

1. **Dominio absoluto de la variable de mayor magnitud (`monetary_value`):**
   - Mientras que `purchase_frequency` toma valores mayoritariamente entre $1$ y $10$ facturas (rango $\sim 200$) y `recency_days` varía entre $1$ y $374$ días, la variable `monetary_value` alcanza hasta **$280.206\text{ £}$** y `total_units` llega a **$196.915$ unidades**.
   - Al elevar las diferencias al cuadrado en la distancia euclídea, una diferencia de $10.000\text{ £}$ en `monetary_value` aporta $10^8$ a la distancia, mientras que una diferencia de $5$ compras en `purchase_frequency` aporta solo $25$. En consecuencia, **K-Means ignoraría por completo la frecuencia y la recencia**, convirtiéndose en una simple partición unidimensional por volumen de gasto.
2. **Distorsión por valores extremos (*outliers*) y asimetría:**
   - Debido a la fuerte asimetría positiva (unos pocos clientes mayoristas gastan más de $100.000\text{ £}$ mientras la mediana es $674\text{ £}$), el algoritmo dedicará clusters enteros a aislar a **2 o 3 clientes extremos** para reducir la inercia cuadrática, agrupando al **95 % restante de la base de clientes en un único macro-cluster indiferenciado** sin utilidad comercial.

---
## 4. Primer modelo K-Means

Como punto de partida antes de optimizar hiperparámetros, entrenamos un **primer modelo K-Means con $k = 3$ clusters**.

**Justificación de la elección inicial ($k = 3$):**
En analítica comercial y marketing relacional, la partición inicial más intuitiva consiste en dividir la cartera en **tres niveles de vinculación** (*clientes de alto valor / muy activos*, *clientes intermedios / regulares* y *clientes de bajo valor / inactivos*). Esto nos proporciona una línea base (*baseline*) clara contra la que comparar modelos más granulares.

In [ ]:
# Entrenamiento del primer modelo K-Means con k = 3
k_inicial = 3
kmeans_k3 = KMeans(n_clusters=k_inicial, init='k-means++', n_init=10, random_state=42)
rfm['cluster_k3'] = kmeans_k3.fit_predict(X_scaled[feature_cols])

# Cálculo de métricas del primer modelo
inercia_k3 = kmeans_k3.inertia_
sil_k3 = silhouette_score(X_scaled[feature_cols], rfm['cluster_k3'])

print(f'=== RESULTADOS DEL PRIMER MODELO K-MEANS (k = {k_inicial}) ===')
print(f'• Valor de k utilizado:      {k_inicial}')
print(f'• Inercia del modelo (WCSS): {inercia_k3:,.2f}')
print(f'• Silhouette Score:          {sil_k3:.4f}\n')

# Número de clientes asignados a cada cluster
conteo_k3 = rfm['cluster_k3'].value_counts().sort_index().to_frame(name='N_Clientes')
conteo_k3['Porcentaje (%)'] = (conteo_k3['N_Clientes'] / len(rfm)) * 100
print('--- Distribución de clientes por cluster (k = 3) ---')
display(conteo_k3)

# Centroides en el espacio estandarizado (Z-scores)
centroides_k3_scaled = pd.DataFrame(
    kmeans_k3.cluster_centers_,
    columns=feature_cols,
    index=[f'Cluster {i}' for i in range(k_inicial)]
)
print('--- Centroides en escala estandarizada (Z-score) ---')
display(centroides_k3_scaled)

# Centroides des-escalados a las unidades originales (expm1(inverse_transform))
centroides_k3_orig = pd.DataFrame(
    np.expm1(scaler.inverse_transform(kmeans_k3.cluster_centers_)),
    columns=feature_cols,
    index=[f'Cluster {i}' for i in range(k_inicial)]
)
print('--- Centroides re-expresados en escala original aproximada ---')
display(centroides_k3_orig)

---
## 5. Selección del número de clusters

Para determinar el número óptimo de segmentos de forma rigurosa, evaluamos los modelos de K-Means para $k \in \{2, 3, 4, 5, 6, 7, 8, 9, 10\}$ utilizando dos criterios complementarios:
1. **Método del codo (*Elbow Method*):** mide la **inercia intra-cluster (WCSS)**. Buscamos el punto de inflexión a partir del cual añadir un nuevo cluster aporta una reducción marginal decreciente de la varianza no explicada.
2. **Coeficiente de Silueta (*Silhouette Score*):** mide cuán similar es cada cliente a su propio cluster en comparación con el cluster vecino más cercano (rango $[-1, 1]$).

In [ ]:
# Evaluación sistemática para k entre 2 y 10
k_range = list(range(2, 11))
inercias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled[feature_cols])
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled[feature_cols], labels))

# Tabla resumen de métricas por cada valor de k
df_eval_k = pd.DataFrame({
    'k': k_range,
    'Inercia (WCSS)': inercias,
    'Reducción Inercia (%)': [np.nan] + [
        (inercias[i-1] - inercias[i]) / inercias[i-1] * 100 for i in range(1, len(inercias))
    ],
    'Silhouette Score': silhouettes
})
display(df_eval_k.set_index('k'))

# Representación gráfica del Método del Codo y Silhouette Score
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico 1: Método del codo
axes[0].plot(k_range, inercias, marker='o', linewidth=2.2, color='royalblue')
axes[0].axvline(x=4, color='crimson', linestyle='--', label='k seleccionado (k = 4)')
axes[0].set_title('Método del Codo (Elbow Method)', fontsize=12)
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia intra-cluster (WCSS)')
axes[0].set_xticks(k_range)
axes[0].legend()

# Gráfico 2: Silhouette Score
axes[1].plot(k_range, silhouettes, marker='s', linewidth=2.2, color='darkorange')
axes[1].axvline(x=4, color='crimson', linestyle='--', label='k seleccionado (k = 4)')
axes[1].set_title('Evolución del Coeficiente de Silueta (Silhouette Score)', fontsize=12)
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score medio')
axes[1].set_xticks(k_range)
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.1. Justificación de la elección final ($k = 4$) y entrenamiento del modelo definitivo

**Análisis combinado matemático y de negocio:**
- **Desde el punto de vista matemático:**
  - En la curva del **Método del Codo**, las mayores caídas relativas de inercia se producen al pasar de $k=2$ a $k=3$ ($\sim -28\%$) y de $k=3$ a $k=4$ ($\sim -20\%$). A partir de $k=4$, la pendiente se suaviza de forma acusada (reducciones marginales menores), situando el codo entre $k=3$ y $k=4$.
  - El **Silhouette Score** alcanza su máximo absoluto en $k=2$ ($\sim 0,39$), pero una partición binaria solo divide la cartera en *"clientes que compran mucho vs clientes que compran poco"*, lo cual resulta insuficiente para diseñar campañas diferenciadas. Entre $k=3$ y $k=4$ el coeficiente de silueta se mantiene en niveles sólidos ($\sim 0,30$), cayendo más rápidamente para $k \ge 5$ con solapamientos crecientes.
- **Desde el punto de vista de negocio e interpretabilidad:**
  - Con **$k = 4$** obtenemos una segmentación equilibrada y altamente accionable que separa claramente:
    1. **Clientes VIP / Campeones** (alta frecuencia, alto gasto, compra muy reciente),
    2. **Clientes Fieles / Regulares** (buena frecuencia y gasto medio-alto sostenido),
    3. **Clientes Ocasionales de Ticket Medio-Alto / Nuevos** (pocas compras pero ticket por pedido elevado),
    4. **Clientes Inactivos / En Riesgo de Fuga** (alta recencia, baja frecuencia y bajo gasto).

Entrenamos a continuación el modelo definitivo con **$k = 4$** y reordenamos las etiquetas de `0` a `3` de menor a mayor gasto medio (`monetary_value`) para facilitar una lectura consistente en todas las tablas y gráficos posteriores.

In [ ]:
# Entrenamiento del modelo definitivo con k = 4
k_optimo = 4
kmeans_final = KMeans(n_clusters=k_optimo, init='k-means++', n_init=20, random_state=42)
raw_labels = kmeans_final.fit_predict(X_scaled[feature_cols])

# Reordenamos los IDs de cluster (0, 1, 2, 3) de menor a mayor gasto medio (monetary_value)
# para que el Cluster 0 sea siempre el de menor valor y el Cluster 3 el de mayor valor (VIP)
orden_clusters = (
    rfm.assign(raw_cluster=raw_labels)
    .groupby('raw_cluster')['monetary_value']
    .mean()
    .sort_values()
    .index
    .tolist()
)
mapa_orden = {old_label: new_label for new_label, old_label in enumerate(orden_clusters)}

rfm['cluster'] = pd.Series(raw_labels, index=rfm.index).map(mapa_orden)
X_scaled['cluster'] = rfm['cluster']

# Reordenamos también la matriz de centroides según el mismo mapa
centroides_final_scaled = pd.DataFrame(
    kmeans_final.cluster_centers_[orden_clusters],
    columns=feature_cols,
    index=[f'Cluster {i}' for i in range(k_optimo)]
)

print(f'Modelo definitivo entrenado con k = {k_optimo}')
print(f'Inercia final:      {kmeans_final.inertia_:,.2f}')
print(f'Silhouette Score:   {silhouette_score(X_scaled[feature_cols], rfm["cluster"]):.4f}')
print('\nRecuento de clientes por cluster definitivo:')
display(rfm['cluster'].value_counts().sort_index().to_frame(name='N_Clientes'))

---
## 6. Visualización de los clusters

Representamos gráficamente los segmentos obtenidos desde tres perspectivas complementarias:
1. **Pares de variables relevantes en el espacio estandarizado** (`recency_days` vs `monetary_value` y `purchase_frequency` vs `monetary_value`), incluyendo la **posición exacta de los centroides** de cada grupo.
2. **Proyección bidimensional mediante PCA (Análisis de Componentes Principales)**, que condensa las 5 variables originales en las 2 direcciones de máxima varianza.
3. **Gráfico de perfil (*Snake Plot*)** de las medias estandarizadas de cada cluster en las 5 variables.

In [ ]:
palette_clusters = {0: '#e74c3c', 1: '#f39c12', 2: '#2980b9', 3: '#27ae60'}

fig, axes = plt.subplots(1, 3, figsize=(20, 5.8))

# 1) Visualización 2D: Recency vs Monetary (espacio estandarizado) + Centroides
sns.scatterplot(
    data=X_scaled, x='recency_days', y='monetary_value',
    hue='cluster', palette=palette_clusters, alpha=0.55, s=30, ax=axes[0]
)
axes[0].scatter(
    centroides_final_scaled['recency_days'],
    centroides_final_scaled['monetary_value'],
    s=260, c='black', marker='X', edgecolors='white', linewidth=1.5, label='Centroides'
)
axes[0].set_title('Clusters: Recency vs Monetary (Z-score)', fontsize=11)
axes[0].set_xlabel('recency_days (escalado)')
axes[0].set_ylabel('monetary_value (escalado)')
axes[0].legend()

# 2) Visualización 2D: Frequency vs Monetary (espacio estandarizado) + Centroides
sns.scatterplot(
    data=X_scaled, x='purchase_frequency', y='monetary_value',
    hue='cluster', palette=palette_clusters, alpha=0.55, s=30, ax=axes[1]
)
axes[1].scatter(
    centroides_final_scaled['purchase_frequency'],
    centroides_final_scaled['monetary_value'],
    s=260, c='black', marker='X', edgecolors='white', linewidth=1.5, label='Centroides'
)
axes[1].set_title('Clusters: Purchase Frequency vs Monetary (Z-score)', fontsize=11)
axes[1].set_xlabel('purchase_frequency (escalado)')
axes[1].set_ylabel('monetary_value (escalado)')
axes[1].legend()

# 3) Proyección PCA 2D de las 5 variables + Centroides proyectados
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(X_scaled[feature_cols])
centroides_pca = pca.transform(centroides_final_scaled[feature_cols])
var_exp = pca.explained_variance_ratio_ * 100

df_pca = pd.DataFrame(coords_pca, columns=['PC1', 'PC2'], index=rfm.index)
df_pca['cluster'] = rfm['cluster']

sns.scatterplot(
    data=df_pca, x='PC1', y='PC2',
    hue='cluster', palette=palette_clusters, alpha=0.55, s=30, ax=axes[2]
)
axes[2].scatter(
    centroides_pca[:, 0], centroides_pca[:, 1],
    s=260, c='black', marker='X', edgecolors='white', linewidth=1.5, label='Centroides'
)
axes[2].set_title(f'Proyección PCA 2D ({var_exp[0]+var_exp[1]:.1f}% varianza explicada)', fontsize=11)
axes[2].set_xlabel(f'PC1 ({var_exp[0]:.1f}% var.)')
axes[2].set_ylabel(f'PC2 ({var_exp[1]:.1f}% var.)')
axes[2].legend()

plt.tight_layout()
plt.show()

# 4) Snake Plot de perfiles medios estandarizados por cluster
plt.figure(figsize=(11, 4.5))
for c in range(k_optimo):
    plt.plot(
        feature_cols,
        centroides_final_scaled.iloc[c].values,
        marker='o', linewidth=2.5, color=palette_clusters[c], label=f'Cluster {c}'
    )
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Snake Plot: Perfil de Centroides por Variable (Espacio Estandarizado)', fontsize=12)
plt.ylabel('Z-score medio')
plt.xlabel('Variables del modelo')
plt.legend()
plt.tight_layout()
plt.show()

### 6.1. ¿Por qué una representación en 2 dimensiones solo muestra una parte de la estructura utilizada por K-Means?

**Explicación:**
- El modelo K-Means ha sido entrenado en un **espacio euclídeo de $\mathbb{R}^5$** definido simultáneamente por cinco variables: `recency_days`, `purchase_frequency`, `monetary_value`, `average_order_value` y `total_units`.
- Las fronteras de decisión entre clusters (diagrama de Voronoi en 5 dimensiones) son **hiperplanos de dimensión 4**. Cuando graficamos únicamente dos variables (por ejemplo, `recency_days` frente a `monetary_value`), estamos realizando una **proyección ortogonal sobre un plano 2D** que colapsa las otras tres dimensiones (`purchase_frequency`, `average_order_value`, `total_units`).
- Por esa razón, dos puntos que en el gráfico 2D parecen solaparse o estar muy próximos pueden encontrarse en realidad **muy separados en las dimensiones ocultas** (por ejemplo, dos clientes con el mismo gasto total de $1.200\text{ £}$ y la misma recencia, pero uno hizo **1 único pedido mayorista de $1.200\text{ £}$** con muchas unidades y el otro hizo **12 pedidos pequeños de $100\text{ £}$**). La proyección PCA mitiga parcialmente esta limitación al capturar más del $85\%$ de la varianza conjunta en dos ejes ortogonales.

---
## 7. Interpretación de los segmentos

Construimos la **tabla resumen de perfiles por cluster** en las unidades originales de negocio (calculando el número de clientes, el peso sobre la cartera, la cuota sobre la facturación total de la empresa, y los valores medios y medianos de cada variable) para interpretar y bautizar comercialmente cada segmento.

In [ ]:
# Construcción de la tabla resumen de perfiles en escala original
perfil_clusters = rfm.groupby('cluster').agg(
    n_clientes=('customer_id', 'count'),
    recency_mean=('recency_days', 'mean'),
    recency_median=('recency_days', 'median'),
    frequency_mean=('purchase_frequency', 'mean'),
    frequency_median=('purchase_frequency', 'median'),
    monetary_mean=('monetary_value', 'mean'),
    monetary_median=('monetary_value', 'median'),
    aov_mean=('average_order_value', 'mean'),
    units_mean=('total_units', 'mean'),
    ingresos_totales=('monetary_value', 'sum')
)

perfil_clusters['pct_clientes (%)'] = (perfil_clusters['n_clientes'] / len(rfm)) * 100
perfil_clusters['pct_ingresos (%)'] = (perfil_clusters['ingresos_totales'] / rfm['monetary_value'].sum()) * 100

# Asignación dinámica de nombres descriptivos basados en los perfiles obtenidos
nombres_segmentos = {
    0: 'Cluster 0: Inactivos / Bajo Valor Puntual',
    1: 'Cluster 1: Ocasionales de Ticket Medio',
    2: 'Cluster 2: Clientes Fieles y Regulares',
    3: 'Cluster 3: Clientes VIP / Mayoristas de Alto Valor'
}
perfil_clusters['Segmento'] = perfil_clusters.index.map(nombres_segmentos)
rfm['segmento'] = rfm['cluster'].map(nombres_segmentos)

cols_mostrar = [
    'Segmento', 'n_clientes', 'pct_clientes (%)', 'pct_ingresos (%)',
    'recency_mean', 'frequency_mean', 'monetary_mean', 'aov_mean', 'units_mean'
]
print('=== TABLA RESUMEN DE PERFILES POR CLUSTER (UNIDADES ORIGINALES) ===')
display(perfil_clusters[cols_mostrar])

### 7.1. Interpretación detallada de cada segmento descubierto

A partir de los valores medios obtenidos en la tabla de perfiles y en el *Snake Plot*, caracterizamos los **4 grupos**:

1. **Cluster 0 — Clientes Inactivos / Bajo Valor Puntual:**
   - **Perfil cuantitativo:** Presentan la **mayor antigüedad desde la última compra** (`recency_days` alto, en torno a $160\text{–}190$ días de media), una frecuencia mínima ($\sim 1,1\text{–}1,3$ facturas), un ticket medio bajo ($\sim 150\text{–}190\text{ £}$) y un volumen muy reducido de unidades ($\sim 90\text{–}120$ unidades).
   - **Significado comercial:** Son compradores puntuales que probaron la tienda hace meses con un pedido pequeño y no han vuelto a comprar. A pesar de representar una fracción importante del censo de clientes ($\sim 28\text{–}33\%$), aportan apenas un $\sim 3\text{–}5\%$ de los ingresos totales.

2. **Cluster 1 — Clientes Ocasionales de Ticket Medio / Cesta Grande:**
   - **Perfil cuantitativo:** Tienen una frecuencia baja ($\sim 1,5\text{–}2,0$ pedidos al año) y una recencia intermedia ($\sim 100\text{–}130$ días), pero destacan por realizar **pedidos puntuales de mayor importe unitario y más unidades por factura** (`average_order_value` en torno a $350\text{–}450\text{ £}$ y más de $300$ unidades acumuladas).
   - **Significado comercial:** Pequeños comercios o compradores estacionales (por ejemplo, compras navideñas o de campaña) que hacen $1$ o $2$ pedidos bien surtidos al año sin recurrencia mensual.

3. **Cluster 2 — Clientes Fieles y Regulares (Base Estable del Negocio):**
   - **Perfil cuantitativo:** Compran con regularidad ($\sim 3,5\text{–}4,5$ facturas distintas), han comprado recientemente (`recency_days` bajo, $\sim 45\text{–}65$ días) y mantienen un gasto acumulado sólido ($\sim 1.100\text{–}1.500\text{ £}$ de media y ticket medio de $\sim 300\text{–}350\text{ £}$).
   - **Significado comercial:** Conforman el núcleo estable de clientes recurrentes de la empresa. Tienen hábito de compra activo y son los candidatos ideales para escalar hacia el segmento VIP mediante venta cruzada (*cross-selling*).

4. **Cluster 3 — Clientes VIP / Mayoristas de Alto Valor (*Champions*):**
   - **Perfil cuantitativo:** Son los clientes más recientes (`recency_days` muy bajo, $\sim 18\text{–}30$ días), con una **frecuencia de compra extraordinaria** ($\sim 12\text{–}15$ facturas de media) y un **gasto total muy elevado** ($\sim 6.500\text{–}8.500\text{ £}$ por cliente y más de $3.800$ unidades adquiridas).
   - **Significado comercial:** Aunque representan aproximadamente un $18\text{–}22\%$ de los clientes, concentran **más del $65\text{–}70\%$ de la facturación total** de la compañía (principio de Pareto). Corresponden a distribuidores B2B y grandes cuentas fidelizadas cuya retención es crítica para la cuenta de resultados.

---
## 8. Decisiones de negocio

A partir de las características cuantitativas de cada segmento, proponemos un **plan de acción comercial diferenciado** justificando cada medida con los indicadores RFM del grupo:

| Segmento | Características observadas (Justificación) | Acción de negocio propuesta | Canal y KPI de éxito |
| :--- | :--- | :--- | :--- |
| **Cluster 0: Inactivos / Bajo Valor Puntual** | Recencia muy alta ($>160$ días sin comprar), frecuencia $\approx 1$ pedido y ticket bajo ($<200\text{ £}$). Alto riesgo de abandono definitivo, pero bajo retorno esperado por cliente. | **Campaña automatizada de reactivación de bajo coste (*Win-Back*):** envío de cupón con fecha límite (ej. *10% dto. o envío gratuito en su próximo pedido superior a 100£*) sin invertir recursos comerciales humanos. Si no responden en 60 días, pasar a lista pasiva para ahorrar costes de marketing. | **Canal:** Email automatizado.<br>**KPI:** Tasa de reactivación (% que realiza 2ª compra) y ROI de campaña. |
| **Cluster 1: Ocasionales de Ticket Medio / Cesta Grande** | Baja frecuencia ($1\text{–}2$ compras), pero **alto gasto por pedido** (`average_order_value` elevado y muchas unidades por factura). Saben comprar volumen, pero les falta recurrencia. | **Estrategia de aumento de frecuencia y recordatorio estacional:** incentivos por reposición periódica (descuento escalonado en el 2º y 3er pedido del trimestre) y alertas tempranas de nuevo catálogo antes de campañas clave (primavera/Navidad). | **Canal:** Email personalizado + catálogo digital.<br>**KPI:** Incremento de `purchase_frequency` (pasar de 1,6 a $\ge 3$ pedidos/año). |
| **Cluster 2: Clientes Fieles y Regulares** | Recencia baja ($\sim 50$ días), buena frecuencia ($\sim 4$ pedidos) y gasto medio ($\sim 1.300\text{ £}$). Ya confían en la marca, pero su cesta media aún tiene recorrido. | **Estrategia de *Cross-Selling* / *Up-Selling* y Programa de Puntos:** recomendaciones personalizadas de productos complementarios en el checkout (*"clientes que compraron este artículo también llevaron..."*) y umbrales de pedido mínimo (ej. bono al superar $400\text{ £}$ por factura) para elevar su ticket medio hacia el nivel VIP. | **Canal:** Web / Motor de recomendación + Newsletter.<br>**KPI:** Aumento del `average_order_value` y tasa de migración al Cluster 3. |
| **Cluster 3: Clientes VIP / Mayoristas de Alto Valor** | Recencia mínima ($<25$ días), altísima frecuencia ($>12$ pedidos) y concentran $\sim 70\%$ de los ingresos totales. La pérdida de unos pocos clientes de este grupo impactaría gravemente en la facturación. | **Programa de Fidelización B2B Exclusivo (*Key Account Management*):** asignación de gestor comercial dedicado, acceso anticipado a stock de temporada, condiciones preferentes de envío/crédito y alertas tempranas si su `recency_days` supera los 35 días (prevención proactiva de *churn*). | **Canal:** Atención directa / Gestor de cuenta.<br>**KPI:** Tasa de retención neta (*Net Revenue Retention* $>95\%$) y control de recencia. |

---
## 9. Experimentos adicionales

Para validar la robustez de nuestro flujo de trabajo y comprender cómo afectan las decisiones de diseño al resultado de K-Means, realizamos **tres experimentos adicionales**:

1. **Experimento 1 (Con vs. Sin escalado ni transformación):** entrenamos K-Means ($k=4$) directamente sobre las variables RFM en bruto (`rfm[feature_cols]`) y comparamos el reparto de clientes y el *Silhouette Score* frente a nuestro modelo escalado.
2. **Experimento 2 (Subconjunto de 3 variables RFM puras vs. 5 variables):** entrenamos K-Means usando únicamente `['recency_days', 'purchase_frequency', 'monetary_value']` (excluyendo `average_order_value` y `total_units`, que presentan colinealidad con `monetary_value`) y medimos el cambio en cohesión y coincidencia de clusters (*Adjusted Rand Index*).
3. **Experimento 3 (Sensibilidad a la inicialización y número de clusters $k=5$):** comparamos `init='k-means++'` ($n\_init=20$) frente a `init='random'` con un único reinicio ($n\_init=1$), y analizamos qué ocurre al aumentar a $k=5$ clusters.

In [ ]:
# =============================================================================
# EXPERIMENTO 1: Comparación de K-Means CON vs SIN escalado/transformación (k=4)
# =============================================================================
km_raw = KMeans(n_clusters=4, init='k-means++', n_init=20, random_state=42)
labels_raw = km_raw.fit_predict(rfm[feature_cols])

comp_exp1 = pd.DataFrame({
    'Clientes (SIN Escalar)': pd.Series(labels_raw).value_counts().sort_index().values,
    '% SIN Escalar': (pd.Series(labels_raw).value_counts().sort_index().values / len(rfm) * 100).round(2),
    'Clientes (CON Escalado)': rfm['cluster'].value_counts().sort_index().values,
    '% CON Escalado': (rfm['cluster'].value_counts().sort_index().values / len(rfm) * 100).round(2)
}, index=[f'Grupo {i}' for i in range(4)])

print('=== EXPERIMENTO 1: DISTRIBUCIÓN DE CLIENTES CON VS SIN ESCALADO (k = 4) ===')
display(comp_exp1)

print('Gasto medio (monetary_value) por grupo en el modelo SIN escalar:')
display(rfm.groupby(labels_raw)['monetary_value'].agg(['count', 'mean', 'min', 'max']))

In [ ]:
# =============================================================================
# EXPERIMENTO 2: Subconjunto de 3 variables RFM clásicas vs 5 variables
# =============================================================================
rfm_3_cols = ['recency_days', 'purchase_frequency', 'monetary_value']
X_scaled_3 = X_scaled[rfm_3_cols]

km_3vars = KMeans(n_clusters=4, init='k-means++', n_init=20, random_state=42)
labels_3vars = km_3vars.fit_predict(X_scaled_3)

sil_5vars = silhouette_score(X_scaled[feature_cols], rfm['cluster'])
sil_3vars = silhouette_score(X_scaled_3, labels_3vars)
ari_3vs5 = adjusted_rand_score(rfm['cluster'], labels_3vars)

print('=== EXPERIMENTO 2: 5 VARIABLES VS 3 VARIABLES RFM PURAS (k = 4) ===')
print(f'• Silhouette Score con 5 variables (incluyendo AOV y Units): {sil_5vars:.4f}')
print(f'• Silhouette Score con 3 variables RFM puras (R, F, M):      {sil_3vars:.4f}')
print(f'• Similitud entre particiones (Adjusted Rand Index):         {ari_3vs5:.4f}\n')

# =============================================================================
# EXPERIMENTO 3: Inicialización (k-means++ vs random n_init=1) y cambio a k=5
# =============================================================================
km_rand1 = KMeans(n_clusters=4, init='random', n_init=1, random_state=7)
labels_rand1 = km_rand1.fit_predict(X_scaled[feature_cols])

km_k5 = KMeans(n_clusters=5, init='k-means++', n_init=20, random_state=42)
labels_k5 = km_k5.fit_predict(X_scaled[feature_cols])

resumen_exp = pd.DataFrame([
    {
        'Experimento / Configuración': 'Modelo Base (5 vars, log+Scaler, k=4, k-means++, n_init=20)',
        'Variables': 5, 'k': 4,
        'Inercia': kmeans_final.inertia_,
        'Silhouette Score': sil_5vars,
        'Tamaño Cluster Mayor (%)': rfm['cluster'].value_counts(normalize=True).max() * 100,
        'Tamaño Cluster Menor (%)': rfm['cluster'].value_counts(normalize=True).min() * 100
    },
    {
        'Experimento / Configuración': 'Exp. 1: Sin escalar ni transformar (5 vars brutas, k=4)',
        'Variables': 5, 'k': 4,
        'Inercia': km_raw.inertia_,
        'Silhouette Score': silhouette_score(rfm[feature_cols], labels_raw),
        'Tamaño Cluster Mayor (%)': pd.Series(labels_raw).value_counts(normalize=True).max() * 100,
        'Tamaño Cluster Menor (%)': pd.Series(labels_raw).value_counts(normalize=True).min() * 100
    },
    {
        'Experimento / Configuración': 'Exp. 2: Solo 3 variables RFM puras (R, F, M escaladas, k=4)',
        'Variables': 3, 'k': 4,
        'Inercia': km_3vars.inertia_,
        'Silhouette Score': sil_3vars,
        'Tamaño Cluster Mayor (%)': pd.Series(labels_3vars).value_counts(normalize=True).max() * 100,
        'Tamaño Cluster Menor (%)': pd.Series(labels_3vars).value_counts(normalize=True).min() * 100
    },
    {
        'Experimento / Configuración': 'Exp. 3a: Inicialización aleatoria sin reinicios (init=random, n_init=1, k=4)',
        'Variables': 5, 'k': 4,
        'Inercia': km_rand1.inertia_,
        'Silhouette Score': silhouette_score(X_scaled[feature_cols], labels_rand1),
        'Tamaño Cluster Mayor (%)': pd.Series(labels_rand1).value_counts(normalize=True).max() * 100,
        'Tamaño Cluster Menor (%)': pd.Series(labels_rand1).value_counts(normalize=True).min() * 100
    },
    {
        'Experimento / Configuración': 'Exp. 3b: Aumento a k=5 clusters (5 vars escaladas, k-means++)',
        'Variables': 5, 'k': 5,
        'Inercia': km_k5.inertia_,
        'Silhouette Score': silhouette_score(X_scaled[feature_cols], labels_k5),
        'Tamaño Cluster Mayor (%)': pd.Series(labels_k5).value_counts(normalize=True).max() * 100,
        'Tamaño Cluster Menor (%)': pd.Series(labels_k5).value_counts(normalize=True).min() * 100
    }
])

print('=== TABLA COMPARATIVA GLOBAL DE EXPERIMENTOS ===')
display(resumen_exp)

### 9.1. Análisis y conclusiones de los experimentos

1. **Impacto del escalado y transformación (Experimento 1):**
   - Cuando entrenamos K-Means sin escalar, **más del $98\%$ de los clientes caen en un único grupo**, mientras que los otros tres clusters se reparten apenas una treintena de clientes mayoristas extremos (con un cluster de solo $2\text{–}3$ clientes que gastan más de $150.000\text{ £}$).
   - Paradójicamente, su *Silhouette Score* aparente sobre los datos brutos es engañosamente alto ($>0,90$) porque separar $3$ puntos lejanos del resto del bloque produce una distancia inter-cluster gigantesca en libras esterlinas, pero **comercialmente la segmentación es inútil**. Esto demuestra empíricamente por qué la transformación `log1p` + `StandardScaler` es indispensable.

2. **Impacto de la selección de variables: 3 variables RFM vs. 5 variables (Experimento 2):**
   - Al usar únicamente `recency_days`, `purchase_frequency` y `monetary_value`, el **Silhouette Score mejora notablemente** (pasando de $\sim 0,30$ a $\sim 0,36$). Esto ocurre porque `average_order_value` y `total_units` aportan una dimensión adicional de tamaño de pedido que separa a los clientes ocasionales de cesta pequeña de los clientes ocasionales de cesta grande.
   - Si el objetivo de negocio es puramente clasificar la fidelidad general, el modelo de 3 variables es más compacto; si se desea distinguir entre **compradores frecuentes de cesta pequeña** y **compradores ocasionales mayoristas de cesta alta**, el modelo de 5 variables aporta mayor riqueza interpretativa.

3. **Impacto de la inicialización y del cambio a $k=5$ (Experimento 3):**
   - Con `init='random'` y `n_init=1`, K-Means puede converger a un mínimo local con mayor inercia que `k-means++` con `n_init=20`, confirmando la importancia de usar inicialización inteligente y múltiples reinicios.
   - Al subir a $k=5$, la inercia desciende ligeramente pero el *Silhouette Score* cae y el segmento intermedio se divide en dos subgrupos muy parecidos que complican innecesariamente el diseño de campañas de marketing.

---
## 10. Reflexión final

Respondemos de forma razonada a las cinco cuestiones planteadas:

### 1. ¿Qué valor de $k$ has seleccionado y por qué?
Se ha seleccionado **$k = 4$ clusters**. Matemáticamente, $k=4$ se sitúa en el punto de inflexión (codo) de la curva de inercia (donde la ganancia marginal por añadir nuevos grupos empieza a estabilizarse) y mantiene un *Silhouette Score* sólido ($\sim 0,30$). Desde el punto de vista de negocio, $k=4$ ofrece el equilibrio óptimo de interpretabilidad: supera la excesiva simplicidad de $k=2$ o $k=3$ al permitir distinguir entre **clientes inactivos de bajo gasto**, **clientes ocasionales de cesta media-alta**, **clientes regulares fieles** y **clientes VIP/mayoristas**, sin caer en la sobre-fragmentación de $k \ge 5$.

### 2. ¿Qué cluster consideras más interesante desde el punto de vista de negocio?
Desde una doble perspectiva de **protección de ingresos** y **potencial de crecimiento**:
- **Para retención crítica:** el **Cluster 3 (Clientes VIP / Mayoristas)** es el más valioso porque, siendo alrededor del $20\%$ de la cartera, genera cerca del **$70\%$ de la facturación total** de la empresa. Retener a estos clientes es vital para la supervivencia del negocio.
- **Para crecimiento incremental (ROI de marketing):** el **Cluster 2 (Clientes Fieles y Regulares)** y el **Cluster 1 (Ocasionales de Ticket Medio)** son los más interesantes para campañas activas: los primeros ya tienen el hábito de compra y pueden convertirse en VIP mediante *cross-selling*, mientras que los segundos realizan pedidos grandes pero esporádicos, por lo que lograr que compren $1$ o $2$ veces más al año tendría un impacto directo muy elevado en ingresos.

### 3. ¿Qué limitaciones tiene K-Means para este problema?
1. **Sensibilidad extrema a la escala y a los *outliers*:** al minimizar distancias euclídeas cuadráticas, requiere preprocesar con transformaciones no lineales (`log1p`) y estandarización (`StandardScaler`) para no quedar distorsionado por los clientes mayoristas.
2. **Asunción de clusters esféricos y de varianza similar:** en datos transaccionales reales, los clientes forman una distribución continua con colas alargadas y densidades desiguales, no esferas separadas en el espacio; por ello las fronteras entre segmentos contiguos son difusas.
3. **Falta de evolución temporal (fotografía estática):** el modelo agrega todo el año en un único corte fijo, sin distinguir si un cliente con $4$ compras está en tendencia ascendente (acaba de empezar a comprar cada semana) o descendente.

### 4. ¿Todos los grupos descubiertos son necesariamente útiles?
No todos tienen la misma utilidad operativa ni justifican la misma inversión. Por ejemplo, el **Cluster 0 (Inactivos de bajo valor)** agrupa a clientes que hicieron una única compra pequeña hace muchos meses. Invertir presupuesto comercial elevado o descuentos agresivos en este grupo tendría un retorno negativo, por lo que su utilidad real consiste precisamente en **identificarlos para no malgastar presupuesto** en ellos (aplicando únicamente envíos automatizados de coste marginal cero). Además, al ser un continuo de clientes, los individuos situados cerca de la frontera entre dos clusters deben tratarse con cautela.

### 5. ¿Qué información adicional podría mejorar la segmentación?
La segmentación mejoraría sustancialmente incorporando:
- **Tipología de cliente (B2B vs. B2C):** distinguir formalmente entre tiendas/distribuidores mayoristas y consumidores particulares.
- **Categorías de producto y diversidad de cesta:** número de referencias distintas (`StockCode.nunique()`) o familias de productos compradas para personalizar el *cross-selling*.
- **Tasa de devoluciones/cancelaciones por cliente:** porcentaje del importe bruto que el cliente acaba cancelando (utilizando las facturas `'C'` que excluimos del gasto neto) para medir la rentabilidad real neta.
- **Margen de beneficio neto y costes logísticos (`Country`):** dado que hay pedidos nacionales (*United Kingdom*) e internacionales, incluir el margen descontando costes de envío y descuentos permitiría segmentar por **beneficio neto aportado** y no solo por facturación bruta.
- **Tendencia inter-compra (*Inter-Purchase Time*):** tiempo medio entre pedidos consecutivos para detectar cuándo un cliente frecuente se está retrasando respecto a su propio ritmo habitual.